# Prompting para clasificación multiclase con Hugging Face

Dataset real: `SetFit/amazon_reviews_multi_es`  
Modelo instructivo: `google/flan-t5-base`

Aquí clasificamos reseñas mediante prompting, sin entrenar un clasificador tradicional.

In [1]:
# !pip install -q transformers datasets accelerate


# 1. Imports

In [4]:
import re
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 2. Cargar datasest 

En este enfoque no hacemos training ni validación. 
Cargamos sólo una pequeña parte del conjunto test. 

In [6]:
# 1. Cargar dataset
dataset = load_dataset("SetFit/amazon_reviews_multi_es")
test_ds = dataset["test"].shuffle(seed=42).select(range(100))
test_ds

Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['id', 'text', 'label', 'label_text'],
    num_rows: 100
})

# 2. Cargar modelo


In [10]:
model_checkpoint = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

# 3. Definir etiquetas

En el dataset las clases son 0, 1, 2, 3, 4. 
- 0 sería 1 estrella.
- 4 sería 5 estrellas
  

In [11]:
labels = ["1 estrella", "2 estrellas", "3 estrellas", "4 estrellas", "5 estrellas"]

id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}

# 4. Prompt más restrictivo


In [12]:
def build_prompt(text):
    return f"""Clasifica la siguiente reseña en una de estas categorías:
            1 estrella
            2 estrellas
            3 estrellas
            4 estrellas
            5 estrellas
            
            Responde SOLO con un número del 1 al 5.
            
            Reseña: {text}
            Respuesta:"""

# 5. Generación


In [ ]:
def generate_label(text, max_new_tokens=5):
    prompt = build_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 6. Debug: inspeccionar algunas salidas


In [17]:
print("=== Debug ===")
for i in range(20):
    text = test_ds[i]["text"]
    output = generate_label(text)
    print("Salida en bruto del modelo:", output)

=== Ejemplos de prompting ===
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: 1 star 2 stars 3
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la
Salida en bruto del modelo: Clasifica la


Observando la salida en crudo del modelo, podemos ver lo siguiente: 
- El modelo está repitiendo el inicio del prompt ("Clasifica la") y no está resolviendo la tarea.
- Otras veces parece que genera texto mezclado "1 star 2 stars 3"

Es decir, 
👉 mezcla inglés + varias clases
👉 no sigue la instrucción


Posibles soluciones: 
- Escribe prompt en inglés (el modelo que stás usando funciona mejor para inglés).
- 

In [23]:
def build_prompt_v2(text):
    return f"""Classify the following Spanish product review.

Possible labels:
1
2
3
4
5

Answer ONLY with one number (1, 2, 3, 4, or 5).

Review: {text}
Answer:"""

In [21]:
def generate_label_v2(text, max_new_tokens=5):
    prompt = build_prompt_v2(text)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Fijate como haciendo únicamente cambiando el idioma del prompt ya funciona: 

In [24]:
print("=== Ejemplos de prompting ===")
for i in range(5):
    text = test_ds[i]["text"]
    output = generate_label_v2(text)
    print("Salida en bruto del modelo:", output)
    

=== Ejemplos de prompting ===
Salida en bruto del modelo: 3
Salida en bruto del modelo: 2
Salida en bruto del modelo: 5
Salida en bruto del modelo: 4
Salida en bruto del modelo: 1


# 7 Normalizar la salida
 
Debemos procesar la salida generada por el modelo y traducirla a las labels. 

In [30]:
def normalize_prediction(pred_text):
    pred_text = pred_text.strip().lower()
    mapping = {
        "1": 0,
        "2": 1,
        "3": 2,
        "4": 3,
        "5": 4,
    }

    if pred_text in mapping:
        return mapping[pred_text]

    return None

Volvemos a inspeccionar la salida. En la celda, únicamente miramos los 5 primeros ejemplos, pero es recomendable analizar más para 1) ver que el formato de la salida siempre es el mismo, y 2) la función de normalización funciona correctamente.

In [33]:
print("=== Debug ===")
for i in range(5):
    text = test_ds[i]["text"]
    output = generate_label_v2(text)
    pred = normalize_prediction(output)

    print("Texto:", text)
    print("Salida del modelo:", output)
    print("Predicción normalizada:", pred)
    # print("Label real:", test_ds[i]["label"])
    print("-" * 80)



=== Debug ===
Texto: COMPRÉ UNA TALLA M, Y SON MUY GRANDES, CORRESPONDEN CON UNA L, EL MATERIAL, TEJIDO Y COLOR, MUY BONITO. LAS VOY A DEVOLVER PARA COGER UNA TALLA MENOS ENCANTADA POR CIERTO CON AMAZON Y SU POLITICA RAPIDISIMA DE DEVOLUCIONES Y ABONOS. UNA MARAVILLA COMPRAR ASI :)
Salida del modelo: 3
Predicción normalizada: 2
--------------------------------------------------------------------------------
Texto: Pues no me ha gustado nada. Es dificil de poner y hay que tener mucho cuidado porque si se humedecen manchan bastante. No se si hacen mucho efecto antimosquito pero de oler huelen bastante.
Salida del modelo: 2
Predicción normalizada: 1
--------------------------------------------------------------------------------
Texto: El artículo cumple con su cometido. Como de costumbre, envío rápido y sin problemas
Salida del modelo: 5
Predicción normalizada: 4
--------------------------------------------------------------------------------
Texto: Preciosos,los he usado para los cajone

# 8. Evaluación


In [42]:
preds = []
refs = []

for example in test_ds:
    output = generate_label_v2(example["text"])
    pred = normalize_prediction(output)
    preds.append(pred)
    refs.append(example["label"])


In [43]:
# Filtramos predicciones que sean None. Necesitamos hacerlo para poder usar sklearn para calcular las métricas
valid_pairs = [(p, r) for p, r in zip(preds, refs) if p is not None]

# ✅ Coverage
coverage = len(valid_pairs) / len(preds)
print(f"El modelos es capaz de generar predicciones para (Coverage): {coverage*100:.2f}%")



El modelos es capaz de generar predicciones para (Coverage): 100.00%


In [44]:
from sklearn.metrics import classification_report, f1_score, accuracy_score
print("=== Classification Report ===")
y_pred = [p for p, _ in valid_pairs]
y_true = [r for _, r in valid_pairs]
if len(y_pred) == 0:
    print("No hay predicciones válidas → revisa prompt/parsing")
else:
    print(classification_report(y_true, y_pred, target_names=labels))

=== Classification Report ===
              precision    recall  f1-score   support

  1 estrella       0.18      0.20      0.19        15
 2 estrellas       0.33      0.25      0.29        20
 3 estrellas       0.26      0.55      0.35        20
 4 estrellas       0.44      0.32      0.37        22
 5 estrellas       0.40      0.17      0.24        23

    accuracy                           0.30       100
   macro avg       0.32      0.30      0.29       100
weighted avg       0.33      0.30      0.29       100



## Ejercicio. Variante few-shot

Prueba ahora a utilizar un enfoque few-shot, es decir, un prompt que incluya ejemplos.

In [46]:
def build_fewshot_prompt(text):
    return f"""Classify the following product review into one of these categories:
                1
                2
                3
                4
                5
                
                Example 1:
                Review: The product is terrible and stopped working on the first day.
                Label: 1
                
                Example 2:
                Review: It's okay, it works, but nothing special.
                Label: 3
                
                Example 3:
                Review: Excellent purchase, fantastic quality and fast delivery.
                Label: 5

                Answer ONLY with one number (1, 2, 3, 4, or 5).
                Review: {text}
                Label:"""
